# COS30049 A2 - Distributions of the scoring methods' recall over many plant seeds (Kaggle CPU session)

Runs `python -m explorer.cli distribution` (`explorer/distribution.py`): the three planted patterns replanted with
100 seeds on DS1 and DS10 and 20 on DS2 (seeds from 70000, disjoint from every tuning and reporting seed), every
frozen method scored on each. Output: `reports/tables/stats_methods_seeds.csv` and `stats_methods_summary.csv`
(mean, spread, 95% interval, paired comparison with the baseline, rank). CPU only, so no GPU quota.

**Settings:** Accelerator **None**, **Internet on**. Then **Save Version -> Save & Run All (Commit)**.

In [ ]:
REPO = "https://github.com/Defaul119/COS30049A2.git"
COMMIT = "fe32e5efd5c37abc21f31efeb87907ddd0f97b12"   # the code this run uses
PINS = ["numpy==2.2.6", "pandas==2.2.3", "scikit-learn==1.6.1", "scipy==1.17.1", "pyarrow==20.0.0", "joblib==1.4.2"]

In [ ]:
import contextlib, hashlib, json, os, shutil, subprocess, sys, threading, time
from importlib.metadata import version
from pathlib import Path

T0 = time.time()                           # the session's clock: Kaggle stops a session 12 h after it starts
SRC = Path("/tmp/COS30049A2")             # the code and the data it works on; not part of the output
WORK = Path("/kaggle/working")             # the notebook's output


def sh(*cmd, log=None, cwd=None, env=None):
    """Run a command, print its output as it comes, keep a copy in WORK/<log>; raise if it fails."""
    with contextlib.ExitStack() as stack:
        copy = stack.enter_context(open(WORK / log, "a", encoding="utf-8")) if log else None
        p = stack.enter_context(subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                 text=True, bufsize=1, env=env))
        for line in p.stdout:
            print(line, end="", flush=True)
            if copy:
                copy.write(line)
                copy.flush()
    if p.returncode:
        raise subprocess.CalledProcessError(p.returncode, cmd)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


# Kaggle grants internet (and GPUs) only to phone-verified accounts, and otherwise switches them off silently.
import socket
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    raise SystemExit("No internet in this session: verify a phone number (kaggle.com -> Settings -> Phone "
                     "verification), switch Internet on in the notebook's Settings panel, and run again.")

if not SRC.exists():
    sh("git", "clone", "--quiet", REPO, str(SRC))
sh("git", "checkout", "--quiet", COMMIT, cwd=SRC)
sh(sys.executable, "-m", "pip", "install", "--quiet", *PINS)
os.chdir(SRC)

In [ ]:
sh(sys.executable, "-m", "explorer.cli", "distribution", log="distribution.log")
tables = WORK / "reports" / "tables"
tables.mkdir(parents=True, exist_ok=True)
for name in ("stats_methods_seeds.csv", "stats_methods_summary.csv"):
    shutil.copy(SRC / "reports" / "tables" / name, tables / name)
shutil.copy(SRC / "reports" / "logs" / "distribution.log", WORK / "distribution_step.log")
(WORK / "kaggle_run.json").write_text(json.dumps({"commit": COMMIT, "hours": round((time.time() - T0) / 3600, 2),
    "packages": {p.split("==")[0]: version(p.split("==")[0]) for p in PINS}}, indent=2), encoding="utf-8")